In [10]:
# CHARGEMENT ET FUSION DES DATASETS GENERES
import json
from collections import Counter

def load_dataset(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        content = content.strip()
        if content.startswith("```json"):
            content = content[7:]
        if content.startswith("```"):
            content = content[3:]
        if content.endswith("```"):
            content = content[:-3]
        data = json.loads(content)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and "dataset" in data:
            return data["dataset"]
        else:
            return [data]
    except Exception as e:
        print(f"Erreur: {e}")
        return []

dataset1 = load_dataset("../dataset/processed/generated.json")
dataset2 = load_dataset("../dataset/processed/generated2.json")
dataset3 = load_dataset("../dataset/processed/generated3.json")

print(f"generated.json: {len(dataset1)}")
print(f"generated2.json: {len(dataset2)}")
print(f"generated3.json: {len(dataset3)}")

all_datasets = dataset1 + dataset2 + dataset3
print(f"Total avant nettoyage: {len(all_datasets)}")

seen_cvs = set()
unique_dataset = []
duplicate_count = 0

for item in all_datasets:
    cv = item.get("cv_text", "")
    cv_signature = cv[:500] if len(cv) > 500 else cv
    if cv_signature not in seen_cvs:
        seen_cvs.add(cv_signature)
        unique_dataset.append(item)
    else:
        duplicate_count += 1

print(f"Doublons supprimés: {duplicate_count}")
print(f"Exemples uniques: {len(unique_dataset)}")

with open("../dataset/processed/complete_dataset.json", "w", encoding="utf-8") as f:
    json.dump({"dataset": unique_dataset}, f, indent=2, ensure_ascii=False)
print("Sauvegarde: complete_dataset.json")

generated.json: 118
generated2.json: 26
generated3.json: 40
Total avant nettoyage: 184
Doublons supprimés: 3
Exemples uniques: 181
Sauvegarde: complete_dataset.json


# 2 Combinaison avec le dataset original

In [11]:
# COMBINAISON AVEC LE DATASET ORIGINAL
import pandas as pd
import json

with open("../dataset/processed/complete_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)
generated_dataset = data.get("dataset", data)
print(f"Dataset genere: {len(generated_dataset)}")

df_original = pd.read_csv("../dataset/processed/resume_jd_match_cleaned.csv")
print(f"Dataset original brut: {len(df_original)}")

bad_scores = [49, 74, 75, 95, 100]
df_filtered = df_original[~df_original["label"].isin(bad_scores)]
df_filtered = df_filtered[(df_filtered["text_clean"].str.len() >= 1000) & 
                          (df_filtered["text_clean"].str.len() <= 10000)]

generic_patterns = [
    "Competences techniques solides", "Excellente maitrise de",
    "Base solide", "Points transferables limites"
]

def has_generic_phrase(text):
    if pd.isna(text):
        return True
    for pattern in generic_patterns:
        if pattern in str(text):
            return True
    return False

df_filtered = df_filtered[~df_filtered["text_clean"].apply(has_generic_phrase)]
print(f"Apres filtres qualite: {len(df_filtered)}")

def convert_original_to_format(row):
    label_to_score = {"Good Fit": 85, "Potential Fit": 60, "No Fit": 25}
    score = label_to_score.get(row["label"], 50)
    
    analysis = f"""SCORE: {score}%

POINTS FORTS:
- Experience pertinente dans le domaine
- Competences techniques alignees

POINTS FAIBLES:
- A approfondir en entretien

VERDICT: Candidat {'recommandé' if score >= 75 else 'potentiel' if score >= 50 else 'non recommande'}"""
    
    return {
        "job_title": row.get("Category", "Non specifie"),
        "job_requirements": "Voir description du poste",
        "job_description": "Poste a pourvoir",
        "cv_text": row["text_clean"],
        "analysis": analysis,
        "score": score
    }

original_converted = [convert_original_to_format(row) for _, row in df_filtered.iterrows()]
print(f"Exemples convertis: {len(original_converted)}")

combined_dataset = generated_dataset + original_converted[:400]
print(f"Dataset combine: {len(combined_dataset)}")

with open("../dataset/processed/combined_dataset.json", "w", encoding="utf-8") as f:
    json.dump({"dataset": combined_dataset}, f, indent=2, ensure_ascii=False)
print("Sauvegarde: combined_dataset.json")

Dataset genere: 181
Dataset original brut: 6240
Apres filtres qualite: 4611
Exemples convertis: 4611
Dataset combine: 581
Sauvegarde: combined_dataset.json


# 3: Equilibrage du dataset (250 par tranche)

In [12]:
# EQUILIBRAGE DU DATASET
import json
import numpy as np
from sklearn.utils import resample

with open("../dataset/processed/combined_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)
dataset = data.get("dataset", data)

scores = [item.get("score", 0) for item in dataset]

low = [item for item in dataset if item.get("score", 0) <= 25]
medium_low = [item for item in dataset if 26 <= item.get("score", 0) <= 50]
medium_high = [item for item in dataset if 51 <= item.get("score", 0) <= 75]
high = [item for item in dataset if item.get("score", 0) >= 76]

print(f"Distribution avant equilibrage:")
print(f"0-25%:   {len(low)}")
print(f"26-50%:  {len(medium_low)}")
print(f"51-75%:  {len(medium_high)}")
print(f"76-100%: {len(high)}")

target = 250

def balance_tranche(items, target):
    if len(items) >= target:
        return resample(items, n_samples=target, random_state=42)
    else:
        sampled = items.copy()
        np.random.seed(42)
        while len(sampled) < target:
            for item in items:
                if len(sampled) >= target:
                    break
                new_item = item.copy()
                old_score = item.get("score", 50)
                variation = np.random.randint(-5, 6)
                new_score = max(0, min(100, old_score + variation))
                new_item["score"] = new_score
                sampled.append(new_item)
        return sampled

balanced_low = balance_tranche(low, target)
balanced_medium_low = balance_tranche(medium_low, target)
balanced_medium_high = balance_tranche(medium_high, target)
balanced_high = balance_tranche(high, target)

balanced_dataset = balanced_low + balanced_medium_low + balanced_medium_high + balanced_high

print(f"\nDataset equilibre: {len(balanced_dataset)}")
print(f"0-25%:   {len(balanced_low)}")
print(f"26-50%:  {len(balanced_medium_low)}")
print(f"51-75%:  {len(balanced_medium_high)}")
print(f"76-100%: {len(balanced_high)}")

with open("../dataset/processed/balanced_dataset.json", "w", encoding="utf-8") as f:
    json.dump({"dataset": balanced_dataset}, f, indent=2, ensure_ascii=False)
print("Sauvegarde: balanced_dataset.json")

Distribution avant equilibrage:
0-25%:   409
26-50%:  21
51-75%:  49
76-100%: 102

Dataset equilibre: 1000
0-25%:   250
26-50%:  250
51-75%:  250
76-100%: 250
Sauvegarde: balanced_dataset.json


# 4: Suppression des experiences irrealistes

In [13]:
# SUPPRESSION DES EXPERIENCES IRREALISTES
import json
import re

with open("../dataset/processed/balanced_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)
dataset = data.get("dataset", data)

print(f"Avant nettoyage: {len(dataset)}")

def has_unrealistic_experience(cv_text):
    patterns = [r'(\d{2,3})\+?\s*(?:ans|annees|years)', r'experience de (\d{2,3})']
    for pattern in patterns:
        matches = re.findall(pattern, cv_text.lower())
        for match in matches:
            try:
                if int(match) > 40:
                    return True
            except:
                pass
    return False

anomalies_index = [i for i, item in enumerate(dataset) if has_unrealistic_experience(item.get("cv_text", ""))]
print(f"Anomalies trouvees: {len(anomalies_index)}")

cleaned_dataset = [item for i, item in enumerate(dataset) if i not in anomalies_index]
print(f"Apres nettoyage: {len(cleaned_dataset)}")

with open("../dataset/processed/cleaned_dataset.json", "w", encoding="utf-8") as f:
    json.dump({"dataset": cleaned_dataset}, f, indent=2, ensure_ascii=False)
print("Sauvegarde: cleaned_dataset.json")

Avant nettoyage: 1000
Anomalies trouvees: 5
Apres nettoyage: 995
Sauvegarde: cleaned_dataset.json


# 5: Conversion finale en JSONL pour fine-tuning

In [14]:
# CONVERSION FINALE EN JSONL POUR FINE-TUNING
import json
from sklearn.model_selection import train_test_split

with open("../dataset/processed/cleaned_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)
dataset = data.get("dataset", data)

print(f"Dataset final: {len(dataset)}")

train, temp = train_test_split(dataset, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

print(f"Train: {len(train)}")
print(f"Validation: {len(val)}")
print(f"Test: {len(test)}")

def save_jsonl(data, filename):
    with open(f"../dataset/processed/{filename}", "w", encoding="utf-8") as f:
        for item in data:
            conv = {
                "messages": [
                    {"role": "system", "content": "Tu es un expert RH specialise dans l'analyse de CV pour le matching avec des offres d'emploi."},
                    {"role": "user", "content": f"OFFRE: {item.get('job_title', 'Non specifie')}\n{item.get('job_requirements', '')}\n\nCV DU CANDIDAT:\n{item.get('cv_text', '')}"},
                    {"role": "assistant", "content": item.get('analysis', '')}
                ]
            }
            f.write(json.dumps(conv, ensure_ascii=False) + "\n")

save_jsonl(train, "train_final.jsonl")
save_jsonl(val, "val_final.jsonl")
save_jsonl(test, "test_final.jsonl")

print("Fichiers JSONL sauvegardes:")
print("   - train_final.jsonl")
print("   - val_final.jsonl")
print("   - test_final.jsonl")
print("Dataset pret pour le fine-tuning QLoRA")

Dataset final: 995
Train: 796
Validation: 99
Test: 100
Fichiers JSONL sauvegardes:
   - train_final.jsonl
   - val_final.jsonl
   - test_final.jsonl
Dataset pret pour le fine-tuning QLoRA
